# Part 3: Advanced Modeling - Ensembles, Tuning, and Full ML Pipeline
This notebook loads the dataset, trains unconstrained and regularized Decision Trees, compares Gini vs. Entropy criteria, evaluates Random Forest and Gradient Boosting, conducts feature ablation, compares CV scores, and tunes parameters via GridSearchCV.

### Load and Prepare Data

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
import joblib

dataframe = pd.read_csv("cleaned_data.csv" if os.path.exists("cleaned_data.csv") else "part1/cleaned_data.csv")
features = dataframe.drop(columns=['customerID', 'ReferralCode', 'MonthlyCharges', 'Churn'])

target_classification = [1 if val == 'Yes' else 0 for val in dataframe['Churn']]
target_classification = pd.Series(target_classification)

encoded_features = features.copy()
contract_mapping = {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
encoded_features['Contract'] = encoded_features['Contract'].map(contract_mapping)
internet_mapping = {'No': 0, 'DSL': 1, 'Fiber optic': 2}
encoded_features['InternetService'] = encoded_features['InternetService'].map(internet_mapping)

nominal_columns = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
                   'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                   'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod']
encoded_features = pd.get_dummies(encoded_features, columns=nominal_columns, drop_first=True)

for col in encoded_features.columns:
    if encoded_features[col].dtype == 'bool':
        encoded_features[col] = encoded_features[col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    encoded_features, target_classification, test_size=0.2, random_state=42
)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Training dataset prepared.")

### Baseline vs. Controlled Decision Tree

In [ ]:
dt_baseline = DecisionTreeClassifier(random_state=42)
dt_baseline.fit(X_train_scaled, y_train)
print("Unconstrained DT - Train Acc: " + str(round(accuracy_score(y_train, dt_baseline.predict(X_train_scaled)), 4)) + 
      " | Test Acc: " + str(round(accuracy_score(y_test, dt_baseline.predict(X_test_scaled)), 4)))

dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_train)
print("Controlled DT - Train Acc: " + str(round(accuracy_score(y_train, dt_controlled.predict(X_train_scaled)), 4)) + 
      " | Test Acc: " + str(round(accuracy_score(y_test, dt_controlled.predict(X_test_scaled)), 4)))

### Gini vs. Entropy & Random Forest

In [ ]:
dt_gini = DecisionTreeClassifier(max_depth=5, criterion='gini', random_state=42)
dt_gini.fit(X_train_scaled, y_train)
dt_entropy = DecisionTreeClassifier(max_depth=5, criterion='entropy', random_state=42)
dt_entropy.fit(X_train_scaled, y_train)

print("Gini Test Accuracy: " + str(round(accuracy_score(y_test, dt_gini.predict(X_test_scaled)), 4)))
print("Entropy Test Accuracy: " + str(round(accuracy_score(y_test, dt_entropy.predict(X_test_scaled)), 4)))

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)
test_auc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test_scaled)[:, 1])
print("RF Test Acc: " + str(round(accuracy_score(y_test, rf_model.predict(X_test_scaled)), 4)) + 
      " | Test AUC: " + str(round(test_auc_rf, 4)))

print("\nTop 5 RF Feature Importances:")
importance_df = pd.DataFrame({'Feature': encoded_features.columns, 'Importance': rf_model.feature_importances_})
print(importance_df.sort_values(by='Importance', ascending=False).head(5))

### Gradient Boosting & Feature Ablation

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_model.fit(X_train_scaled, y_train)
print("GB Test Acc: " + str(round(accuracy_score(y_test, gb_model.predict(X_test_scaled)), 4)) + 
      " | Test AUC: " + str(round(roc_auc_score(y_test, gb_model.predict_proba(X_test_scaled)[:, 1]), 4)))

In [ ]:
lowest_5 = importance_df.sort_values(by='Importance').head(5)['Feature'].tolist()
print("Ablating (removing) lowest importance columns: " + str(lowest_5))

X_train_ablated = pd.DataFrame(X_train_scaled, columns=encoded_features.columns).drop(columns=lowest_5)
X_test_ablated = pd.DataFrame(X_test_scaled, columns=encoded_features.columns).drop(columns=lowest_5)

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_ablated, y_train)
print("Full RF AUC: " + str(round(test_auc_rf, 5)))
print("Reduced RF AUC: " + str(round(roc_auc_score(y_test, rf_reduced.predict_proba(X_test_ablated)[:, 1]), 5)))

### Cross-Validated Comparison

In [ ]:
cv_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models_dict = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', C=1.0, max_iter=1000, random_state=42),
    'Controlled Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
}
for name, model in models_dict.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv_split, scoring='roc_auc')
    print(name + " - CV Mean AUC: " + str(round(scores.mean(), 4)) + " | Std: " + str(round(scores.std(), 4)))

### Pipeline Hyperparameter Tuning (GridSearchCV)

In [ ]:
pipeline_rf = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)
parameters_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}
search_engine = GridSearchCV(pipeline_rf, parameters_grid, cv=cv_split, scoring='roc_auc', n_jobs=-1)
search_engine.fit(X_train, y_train)
print("Best Parameters: " + str(search_engine.best_params_))
print("Best CV Score (AUC): " + str(round(search_engine.best_score_, 4)))
best_pipeline_model = search_engine.best_estimator_

### Manual Learning Curve Table

In [ ]:
training_sizes = [0.2, 0.4, 0.6, 0.8, 1.0]
print("Fraction | Train AUC | Test AUC")
print("---------------------------------")
for fraction in training_sizes:
    subset_size = int(fraction * len(X_train))
    X_subset = X_train.iloc[0:subset_size]
    y_subset = y_train.iloc[0:subset_size]
    best_pipeline_model.fit(X_subset, y_subset)
    
    if len(np.unique(y_subset)) >= 2:
        tr_auc = roc_auc_score(y_subset, best_pipeline_model.predict_proba(X_subset)[:, 1])
    else:
        tr_auc = 1.0
    te_auc = roc_auc_score(y_test, best_pipeline_model.predict_proba(X_test)[:, 1])
    print(str(round(fraction, 1)) + "      | " + str(round(tr_auc, 4)) + "    | " + str(round(te_auc, 4)))

### Model Serialization & Verification

In [ ]:
joblib.dump(best_pipeline_model, "../best_model.pkl" if os.path.exists("../best_model.pkl") else "best_model.pkl")
print("Model saved to best_model.pkl")

reloaded_model = joblib.load("../best_model.pkl" if os.path.exists("../best_model.pkl") else "best_model.pkl")
test_samples = X_test.iloc[[0, 1]]
print("Reloaded Model Predictions: " + str(reloaded_model.predict(test_samples)))
print("Reloaded Model Probabilities: " + str(reloaded_model.predict_proba(test_samples)[:, 1]))